# 003 · Readout amplitude sweep

**Level:** medium · **Topics:** Gen2, SHFQA, readout, sweeps, chunking · **Type:** fix the bugs

A customer wants to find the best readout power, so they sweep the amplitude of the readout pulse.
They copied the sweep from their amplitude Rabi experiment, where the same code works on the drive line.

On the readout line the experiment **does not compile**.

Find out why, and fix it in two different ways. Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.simple import *

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Pulse and sweep

📖 [Pulse library](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/01_pulse_library.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
readout_pulse = pulse_library.const(length=1e-6, amplitude=1.0)

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=10)

## Experiment

This is the customer's code.

📖 [Measurement rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/03_measurement_rules.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html) ·
[Experiment chunking](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/tutorials/01_experiment_chunking.html)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def readout_amplitude_sweep():
    with dsl.acquire_loop_rt(count=16):
        with dsl.sweep(parameter=amplitudes) as amplitude:
            with dsl.section(name="readout"):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    measure_pulse_amplitude=amplitude,
                    integration_kernel=readout_pulse,
                    handle="result",
                    reset_delay=1e-6,
                )


experiment = readout_amplitude_sweep()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Compile and run

📖 [Experiment](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check the result

One point per amplitude is expected.

📖 [Results](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/06_results/concepts/index.html)

In [ ]:
# check
print(results["result"].data.shape)
print(results["result"].axis)

**Expected result:** shape `(10,)` and an axis from 0.1 to 1.0.

## Questions

1. Read the error message. Why can the drive line sweep an amplitude in real time, and the readout line not?
2. **Fix A:** make the amplitude sweep a near-time sweep.
3. **Fix B:** keep the sweep inside the real-time loop and use chunking. Try `chunk_count=2` and `chunk_count=5` first. Which value works, and why?
4. Is there a difference between fix A and fix B in what the instrument does?

<details>
<summary>Hint 1</summary>

A sweep is real-time when it is inside `acquire_loop_rt`, and near-time when it is outside.
</details>

<details>
<summary>Hint 2</summary>

Chunking splits one real-time sweep into several real-time runs. How many different readout waveforms may one run contain?
</details>

The solution is in [solution.ipynb](solution.ipynb).